In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
model = efficientnet_v2_s(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 26) # 26 classes (A-Z)

In [ ]:
# Write your code here
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct = 0.0, 0
    for images, labels in train_loader:
        labels = labels- 1
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
    return running_loss / len(train_loader), 100 * correct / len(train_loader.dataset)


In [ ]:
# Write your code here
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct = 0.0, 0
    for images, labels in train_loader:
        labels = labels - 1
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
    return running_loss / len(train_loader), 100 * correct / len(train_loader.dataset)

In [ ]:
# Write your code here
def validate_tta(model, test_loader, criterion, device):
    model.eval()
    running_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            labels = labels - 1
            images, labels = images.to(device), labels.to(device)

            orig = model(images)
            h_flip = model(torch.flip(images, dims=[3]))
            v_flip = model(torch.flip(images, dims=[2]))

            avg_preds = (orig + h_flip + v_flip) / 3
            loss = criterion(avg_preds, labels)
            running_loss += loss.item()
            _, predicted = torch.max(avg_preds, 1)
            correct += (predicted == labels).sum().item()
    return running_loss / len(test_loader), 100 * correct / len(test_loader.dataset)
